### General Information Scraping

#### Target URLs

| # | Page | URL |
|---|------|-----|
| 1 | Academic Calendar | `https://www.hof-university.com/studying-at-hof-university/our-degree-programs/academic-calendar.html` |
| 2 | How to Finance Your Studies | `https://www.hof-university.com/studying-at-hof-university/preparing-your-stay/how-to-finance-your-studies.html` |
| 3 | Application and Admission | `https://www.hof-university.com/studying-at-hof-university/application-and-admission.html` |
| 4 | FAQs for Degree-Seeking Students | `https://www.hof-university.com/studying-at-hof-university/preparing-your-stay/faqs-for-degree-seeking-students.html` |


#### Scraping Approach

These pages are scraped using **static HTML parsing** (`requests` + `BeautifulSoup`), as their content does not require JavaScript rendering.



In [2]:
import requests
from bs4 import BeautifulSoup
import json
import re
from typing import Dict, List, Any, Optional
import os
import json

class HofUniversityScraper:
    """Scraper for extracting data from Hof University pages"""
    
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)
    
    def fetch_page(self, url: str) -> Optional[BeautifulSoup]:
        """Fetch and parse webpage"""
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.RequestException as e:
            print(f"Error fetching {url}: {e}")
            return None
    
    def extract_academic_calendar(self, soup: BeautifulSoup) -> List[Dict]:
        """Extract academic calendar data from the page"""
        calendars = []
        
        # Find all semester containers
        semester_containers = soup.find_all('div', class_='frame-group-container')
        
        for container in semester_containers:
            semester_data = {}
            
            # Get semester title
            header = container.find('h2')
            if header:
                title = header.get_text(strip=True)
                # Skip the main page header "Academic calendar"
                if title.lower() == 'academic calendar':
                    continue
                semester_data['semester_name'] = title
            
            # Extract table data
            table = container.find('table')
            if table:
                semester_data['dates'] = []
                rows = table.find_all('tr')
                
                for row in rows:
                    cells = row.find_all('td')
                    if len(cells) >= 2:
                        # Clean the text content
                        label_td = cells[0]
                        value_td = cells[1]
                        
                        # Extract all text from label cell (may contain multiple p tags)
                        label_parts = []
                        for elem in label_td.find_all(['strong', 'p']):
                            text = elem.get_text(strip=True)
                            if text:
                                label_parts.append(text)
                        
                        # If no nested elements, get direct text
                        if not label_parts:
                            label_text = label_td.get_text(strip=True)
                            if label_text:
                                label_parts.append(label_text)
                        
                        # Extract all text from value cell
                        value_parts = []
                        for elem in value_td.find_all('p'):
                            text = elem.get_text(strip=True)
                            if text and not text.startswith('The individual'):
                                value_parts.append(text)
                        
                        # If no p tags, get direct text
                        if not value_parts:
                            value_text = value_td.get_text(strip=True)
                            if value_text and value_text != '\xa0':
                                value_parts.append(value_text)
                        
                        # Add each label-value pair
                        for i, label in enumerate(label_parts):
                            if i < len(value_parts):
                                semester_data['dates'].append({
                                    label: value_parts[i]
                                })
            
            if semester_data:
                calendars.append(semester_data)
        
        return calendars
    
    def extract_financing_info(self, soup: BeautifulSoup) -> Dict:
        """Extract financing and scholarship information"""
        data = {
            'title': '',
            'expenses': {
                'monthly': [],
                'per_semester': [],
                'one_time': []
            },
            'working_during_studies': {},
            'scholarships': {}
        }
        
        # Get page title
        h1 = soup.find('h1')
        if h1:
            data['title'] = h1.get_text(strip=True)
        
        # Extract expense tables
        frame_containers = soup.find_all('div', class_='frame-group-container')
        
        for container in frame_containers:
            # Check for expense chart header
            header = container.find('h2')
            if header and 'expenses chart' in header.get_text(strip=True).lower():
                # Find all sub-sections (monthly, per semester, one-time)
                h4_headers = container.find_all('h4')
                tables = container.find_all('table')
                
                for i, h4 in enumerate(h4_headers):
                    header_text = h4.get_text(strip=True).lower()
                    if i < len(tables):
                        table = tables[i]
                        table_data = []
                        rows = table.find_all('tr')
                        
                        for row in rows:
                            cells = row.find_all(['td', 'th'])
                            if cells:
                                if len(cells) >= 3:
                                    expense = cells[0].get_text(strip=True)
                                    amount = cells[1].get_text(strip=True)
                                    info = cells[2].get_text(strip=True)
                                    
                                    if expense and expense not in ['Expense for', 'Expense\xa0for', 'Strong']:
                                        # Combine amount and info if info exists
                                        if info and info != '\xa0':
                                            value = f"{amount}, {info}"
                                        else:
                                            value = amount
                                        
                                        table_data.append({
                                            expense: value
                                        })
                                elif len(cells) == 2:
                                    expense = cells[0].get_text(strip=True)
                                    amount = cells[1].get_text(strip=True)
                                    
                                    if expense and expense not in ['Expense for', 'Expense\xa0for', 'Strong']:
                                        table_data.append({
                                            expense: amount
                                        })
                        
                        if 'monthly' in header_text:
                            data['expenses']['monthly'] = table_data
                        elif 'per semester' in header_text or 'semester' in header_text:
                            data['expenses']['per_semester'] = table_data
                        elif 'one-time' in header_text or 'one time' in header_text:
                            data['expenses']['one_time'] = table_data
        
        # Extract "Working during your studies" section
        working_section = soup.find('div', class_='scholarshipteaser')
        if working_section:
            working_title = working_section.find('h2')
            if working_title:
                data['working_during_studies']['title'] = working_title.get_text(strip=True)
            
            working_content = []
            for p in working_section.find_all('p'):
                text = p.get_text(strip=True)
                if text:
                    working_content.append(text)
            
            if working_content:
                data['working_during_studies']['content'] = '\n'.join(working_content)
        
        # Extract scholarships - flat dictionary structure
        # 1. Find the textpic section with introduction
        textpic_sections = soup.find_all('div', class_='textpic-text')
        for section in textpic_sections:
            text_content = section.get_text(strip=True).lower()
            if 'hof university offers a limited number of scholarships' in text_content:
                intro_parts = []
                
                for p in section.find_all('p'):
                    text = p.get_text(strip=True)
                    if text:
                        intro_parts.append(text)
                
                for ul in section.find_all('ul'):
                    for li in ul.find_all('li'):
                        intro_parts.append(f"• {li.get_text(strip=True)}")
                
                data['scholarships']['Scholarships for degree-seeking international students'] = '\n'.join(intro_parts)
                break
        
        # 2. Find individual scholarship type sections (Germany Scholarship, Senior Students, etc.)
        for container in frame_containers:
            header = container.find('h2')
            if header:
                header_text = header.get_text(strip=True)
                # Look for specific scholarship sections
                if any(keyword in header_text.lower() for keyword in ['germany scholarship', 'deutschlandstipendium', 'scholarships for senior']):
                    frame_inner = container.find('div', class_='frame-inner')
                    if frame_inner:
                        content_parts = []
                        
                        # Extract all paragraphs
                        for p in frame_inner.find_all('p', recursive=False):
                            text = p.get_text(strip=True)
                            if text:
                                content_parts.append(text)
                        
                        # Extract lists
                        for ul in frame_inner.find_all('ul', recursive=False):
                            for li in ul.find_all('li'):
                                content_parts.append(f"• {li.get_text(strip=True)}")
                        
                        # Extract h4 subheadings and their content
                        h4_headers = frame_inner.find_all('h4', recursive=False)
                        for h4 in h4_headers:
                            content_parts.append(f"\n**{h4.get_text(strip=True)}**")
                            
                            # Get content after h4
                            current = h4.find_next_sibling()
                            while current and current.name not in ['h2', 'h3', 'h4']:
                                if current.name == 'p':
                                    text = current.get_text(strip=True)
                                    if text:
                                        content_parts.append(text)
                                elif current.name == 'ul':
                                    for li in current.find_all('li'):
                                        content_parts.append(f"• {li.get_text(strip=True)}")
                                current = current.find_next_sibling()
                        
                        if content_parts:
                            data['scholarships'][header_text] = '\n'.join(content_parts)
        
        return data
    
    def extract_admission_info(self, soup: BeautifulSoup) -> Dict[str, str]:
        """Extract admission related content"""
        data = {}
        
        # Extract "5 Steps to apply Hof University" section
        five_steps_data = self.extract_five_steps_section(soup)
        if five_steps_data:
            data['5 Steps to apply Hof University'] = five_steps_data
            
            # Extract VPD needed programs from "Prepare your documents" step
            vpd_programs = []
            for step in five_steps_data:
                if 'Prepare your documents' in step.get('step_title', ''):
                    content = step.get('content', '')
                    # Look for the master's programs list after "apply for one of the following master's programs:"
                    lines = content.split('\n')
                    in_programs_section = False
                    for line in lines:
                        # Start capturing when we hit the programs section
                        if 'apply for one of the following master' in line.lower():
                            in_programs_section = True
                            continue
                        # Extract programs - looking for nested items with ◦
                        if in_programs_section:
                            stripped = line.strip()
                            if stripped.startswith('◦'):
                                # Extract program name, removing the ◦ symbol
                                program = stripped[1:].strip()
                                if program and program not in vpd_programs:  # Avoid duplicates
                                    vpd_programs.append(program)
                            elif stripped.startswith('•') and 'you did not complete' not in stripped:
                                # Stop when we hit regular bullet points that aren't part of the nested list
                                break
            
            if vpd_programs:
                data['VPD_needed_programs'] = vpd_programs


        # Find all frame containers with h2 headers
        frame_containers = soup.find_all('div', class_='frame-group-container')

        # List of H2 headers whose content must be split into multiple dictionary entries
        SPLIT_HEADERS = ["Language skills", "Enrollment & Campus Card"]
        
        # Mapping from the H4 subheading text (in the HTML source) to the desired final key
        key_mapping = {
            "Applying for a program taught in English language": "language requirements for english taught programs",
            "Applying for a program taught in German language": "language requirements for german taught programs",
            "Applying for a program taught in German language*": "language requirements for german taught programs",
            # NEW: Enrollment and Campus Card Mappings
            "Enrollment": "Enrollment",
            "Campus Card": "Campus Card"
        }
        
        
        for container in frame_containers:
            header = container.find('h2')
            if not header:
                continue
                
            header_text_raw = header.get_text(strip=True)
            bodytext = container.find('div', class_='scholarship-bodytext')
            
            if not bodytext:
                continue

            # 1. Logic to rename the main headers
            header_text_final = header_text_raw
            if header_text_raw == "Application for bachelor's programs":
                header_text_final = "basic application requirements of bachelor's programs"
            elif header_text_raw == "Application for master's programs":
                header_text_final = "basic application requirements of master's programs"
            
            
            # 2. SPECIAL LOGIC for sections that require content splitting
            if header_text_raw in SPLIT_HEADERS:
                temp_data = {}
                current_key = None
                
                # Iterate through all children of the bodytext to separate content blocks
                for elem in bodytext.children:
                    if not hasattr(elem, 'name'):
                        continue
                    
                    # Check for H4 as a new subsection start
                    if elem.name == 'h4':
                        h4_text = elem.get_text(strip=True)
                        if h4_text in key_mapping:
                            current_key = key_mapping[h4_text]
                            temp_data[current_key] = [] # Initialize content list for the new key
                            continue
                        
                    elif current_key:
                        # Collect paragraphs and lists under the currently set key
                        if elem.name == 'p':
                            text = elem.get_text(strip=True)
                            if text:
                                temp_data[current_key].append(text)
                        
                        elif elem.name == 'ul':
                            for li in elem.find_all('li'):
                                temp_data[current_key].append(f"• {li.get_text(strip=True)}")
                    
                
                # Store the split content in the main data dictionary
                for key, parts in temp_data.items():
                    data[key] = '\n\n'.join(parts)
                
                continue

            # 3. DEFAULT EXTRACTION LOGIC (for all other headers that were not split)

            content_parts = []
            skip_first_question = True  # Flag to skip the first paragraph with the question
            
            # Extract all content in order
            for elem in bodytext.children:
                if not hasattr(elem, 'name'):
                    continue
                
                if elem.name == 'p':
                    # Check if this is the first paragraph with a question (has strong tag and ends with ?)
                    strong = elem.find('strong')
                    text = elem.get_text(strip=True)
                    
                    if skip_first_question and strong and '?' in text:
                        skip_first_question = False
                        continue
                    
                    if text:
                        # Add line break before paragraphs that look like new sections
                        if content_parts and any(keyword in text for keyword in ['Application for', 'We recommend', 'For application', 'Additionally']):
                            content_parts.append('')  # Empty line for spacing
                        content_parts.append(text)
                
                elif elem.name == 'h4':
                    # Add h4 as subheading with line break before
                    content_parts.append(f"\n**{elem.get_text(strip=True)}**")
                
                elif elem.name == 'ul':
                    for li in elem.find_all('li'):
                        content_parts.append(f"• {li.get_text(strip=True)}")
            
            # Store with the final (potentially renamed) header as key, content as value
            if content_parts:
                data[header_text_final] = '\n'.join(content_parts)
        
        return data

        
       
    
    def extract_five_steps_section(self, soup: BeautifulSoup) -> List[Dict]:
        """Extract the '5 Steps to apply Hof University' accordion section"""
        steps = []
        
        # Look for the frame container with "5 Steps to Hof University" header
        frame_containers = soup.find_all('div', class_='frame-container')
        
        for container in frame_containers:
            # Check if this container has the "5 Steps" header
            header = container.find('h2')
            if header and '5 Steps' in header.get_text(strip=True):
                # Find the accordion within this container
                accordion = container.find('div', class_='accordion')
                if accordion:
                    # Extract each accordion item (step)
                    accordion_items = accordion.find_all('div', class_='accordion-item')
                    
                    for item in accordion_items:
                        step_data = {}
                        
                        # Get step title from accordion header button
                        accordion_header = item.find('h4', class_='accordion-header')
                        if accordion_header:
                            button = accordion_header.find('button')
                            if button:
                                # Clean the title (remove accordion opener span)
                                title = button.get_text(strip=True)
                                title = title.replace('☰', '').strip()
                                step_data['step_title'] = title
                        
                        # Get step content from accordion body
                        accordion_body = item.find('div', class_='accordion-body')
                        if accordion_body:
                            content_parts = []
                            
                            # Extract paragraphs
                            for p in accordion_body.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            
                            # Extract section headers (h4 inside accordion)
                            for h4 in accordion_body.find_all('h4'):
                                header_text = h4.get_text(strip=True)
                                if header_text:
                                    content_parts.append(f"\n**{header_text}**")
                            
                            # Extract lists
                            for ul in accordion_body.find_all('ul'):
                                for li in ul.find_all('li'):
                                    li_text = li.get_text(strip=True)
                                    if li_text:
                                        # Check for nested lists
                                        nested_ul = li.find('ul')
                                        if nested_ul:
                                            # Extract main list item text without nested items
                                            main_text = li.contents[0] if li.contents else ''
                                            if isinstance(main_text, str):
                                                content_parts.append(f"• {main_text.strip()}")
                                            # Extract nested items
                                            for nested_li in nested_ul.find_all('li'):
                                                nested_text = nested_li.get_text(strip=True)
                                                if nested_text:
                                                    content_parts.append(f"  ◦ {nested_text}")
                                        else:
                                            content_parts.append(f"• {li_text}")
                            
                            if content_parts:
                                step_data['content'] = '\n'.join(content_parts)
                        
                        if step_data:
                            steps.append(step_data)
                
                break  # We found the 5 Steps section, no need to continue
        
        return steps
    
    
    def extract_faqs(self, soup: BeautifulSoup) -> Dict[str, str]:
        """Extract FAQ information in simple Q:A format"""
        faqs = {}
        seen_questions = set()
        
        # Question to exclude
        EXCLUDED_QUESTION = "What requirements do I have to fulfill in order to get an admission?"
        
        def clean_question(question_text: str) -> str:
            """Remove 'last modified' text from questions"""
            
            cleaned = re.sub(r'\s*\(last modified:?[^)]*\)\s*$', '', question_text, flags=re.IGNORECASE)
            return cleaned.strip()
        
        # Method 1: Look for accordion structure (most common on Hof University site)
        accordions = soup.find_all('div', class_='accordion')
        
        for accordion in accordions:
            # Find all accordion items
            accordion_items = accordion.find_all('div', class_='accordion-item')
            
            for item in accordion_items:
                # Get question from button inside accordion-header
                header = item.find('h4', class_='accordion-header')
                if header:
                    button = header.find('button')
                    if button:
                        question = button.get_text(strip=True)
                        # Remove the accordion opener span text if present
                        question = question.replace('☰', '').strip()
                        # Clean the last modified text
                        question = clean_question(question)
                        
                        # Skip if this is the excluded question
                        if question == EXCLUDED_QUESTION:
                            continue
                        
                        # Get answer from accordion-body
                        body = item.find('div', class_='accordion-body')
                        if body:
                            answer_parts = []
                            
                            # Extract paragraphs
                            for p in body.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    answer_parts.append(text)
                            
                            # Extract lists
                            for ul in body.find_all(['ul', 'ol']):
                                for li in ul.find_all('li'):
                                    answer_parts.append(f"• {li.get_text(strip=True)}")
                            
                            if answer_parts and question not in seen_questions:
                                faqs[question] = '\n'.join(answer_parts)
                                seen_questions.add(question)
        
        # Method 2: Fallback - Look for h2 sections if no accordions found
        if not faqs:
            h2_sections = soup.find_all('h2')
            
            for h2 in h2_sections:
                section_title = h2.get_text(strip=True)
                
                # Look for h3/h4 questions under this h2
                current = h2.find_next_sibling()
                
                while current and current.name != 'h2':
                    if current.name in ['h3', 'h4']:
                        header_text = current.get_text(strip=True)
                        # Clean the last modified text
                        header_text = clean_question(header_text)
                        
                        # Skip if this is the excluded question
                        if header_text == EXCLUDED_QUESTION:
                            current = current.find_next_sibling()
                            continue
                        
                        # Only process if it's a question
                        if '?' in header_text and header_text not in seen_questions:
                            answer_parts = []
                            answer_elem = current.find_next_sibling()
                            
                            while answer_elem and answer_elem.name not in ['h2', 'h3', 'h4']:
                                if answer_elem.name == 'p':
                                    text = answer_elem.get_text(strip=True)
                                    if text:
                                        answer_parts.append(text)
                                elif answer_elem.name in ['ul', 'ol']:
                                    for li in answer_elem.find_all('li'):
                                        answer_parts.append(f"• {li.get_text(strip=True)}")
                                
                                answer_elem = answer_elem.find_next_sibling()
                            
                            if answer_parts:
                                faqs[header_text] = '\n'.join(answer_parts)
                                seen_questions.add(header_text)
                    
                    current = current.find_next_sibling()
        
        return faqs
        
    def scrape_all_pages(self, urls: List[str]) -> Dict[str, Any]:
        """Main method to scrape all provided URLs"""
        all_data = {}
        
        for url in urls:
            print(f"Scraping: {url}")
            soup = self.fetch_page(url)
            
            if not soup:
                print(f"Failed to fetch {url}")
                continue
            
            # Determine page type and extract accordingly
            if 'academic-calendar' in url:
                all_data['academic_calendar'] = self.extract_academic_calendar(soup)
            elif 'finance' in url or 'financing' in url:
                all_data['financing'] = self.extract_financing_info(soup)
            elif 'application' in url or 'admission' in url:
                all_data['general_admission_related_info'] = self.extract_admission_info(soup)
            elif 'faq' in url.lower():
                all_data['faqs'] = self.extract_faqs(soup)
            else:
                # Generic extraction
                all_data[url.split('/')[-1].replace('.html', '')] = self.extract_financing_info(soup)
        
        return all_data
    
    def save_json(self, data: Dict, filename: str = 'data/general_hof_university_data.json'):
        """Save extracted data to JSON file"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f"Data saved to {filename}")
        return filename


# Main execution
if __name__ == "__main__":
    urls = [
        "https://www.hof-university.com/studying-at-hof-university/our-degree-programs/academic-calendar.html",
        "https://www.hof-university.com/studying-at-hof-university/preparing-your-stay/how-to-finance-your-studies.html",
        "https://www.hof-university.com/studying-at-hof-university/application-and-admission.html",
        "https://www.hof-university.com/studying-at-hof-university/preparing-your-stay/faqs-for-degree-seeking-students.html"
    ]
    
    os.makedirs("data", exist_ok=True)

    scraper = HofUniversityScraper()
    extracted_data = scraper.scrape_all_pages(urls)
    
    # Save to JSON
    output_file = scraper.save_json(extracted_data)

Scraping: https://www.hof-university.com/studying-at-hof-university/our-degree-programs/academic-calendar.html
Scraping: https://www.hof-university.com/studying-at-hof-university/preparing-your-stay/how-to-finance-your-studies.html
Scraping: https://www.hof-university.com/studying-at-hof-university/application-and-admission.html
Scraping: https://www.hof-university.com/studying-at-hof-university/preparing-your-stay/faqs-for-degree-seeking-students.html
Data saved to data/general_hof_university_data.json
